# π0.5 Action Expert q/k/v/o LoRA 比較実験

合格済みの q/v LoRA 提出物を変更せず、Action Expert の Attention 対象だけを q/k/v/o へ広げた候補を別の Drive ディレクトリで学習します。公平な比較のため、新しい Colab ランタイムで元の公開 π0.5 から開始してください。


In [ ]:
from pathlib import Path
import json
import os
import shutil
import subprocess

REPO_URL = "https://github.com/KosukeKomeya/PARC2026_pre.git"
BRANCH = "feature/experiment"
REPO_DIR = Path("/content/PARC2026_pre")

if (REPO_DIR / ".git").is_dir():
    subprocess.run(["git", "pull", "--ff-only", "origin", BRANCH], cwd=REPO_DIR, check=True)
elif REPO_DIR.exists():
    raise RuntimeError(f"既存パスがGitリポジトリではありません: {REPO_DIR}")
else:
    subprocess.run(["git", "clone", "--depth", "1", "--branch", BRANCH, REPO_URL, str(REPO_DIR)], check=True)

print(subprocess.check_output(["git", "log", "-1", "--oneline"], cwd=REPO_DIR, text=True).strip())
%cd /content/PARC2026_pre


## 元モデルと固定ランタイムを準備

PaliGemma の利用条件へ同意済みの Hugging Face read token を使用します。セットアップ済みランタイムを再利用する場合も、q/v 統合モデルで元モデルが置換されていないことを検査します。


In [ ]:
%pip install -q "huggingface-hub>=0.34.2,<0.36.0"
from huggingface_hub import notebook_login
notebook_login()


In [ ]:
PI05_PYTHON = Path("/content/pi05_py310/bin/python")
if PI05_PYTHON.is_file():
    setup_command = ["python", "examples/pi05_parc_colab_setup.py", "--reuse-runtime", "--skip-download"]
else:
    setup_command = ["python", "examples/pi05_parc_colab_setup.py"]
subprocess.run(setup_command, cwd=REPO_DIR, check=True)

BASE_MODEL_DIR = REPO_DIR / "submission_template/model_weights/pi05_libero_finetuned_v044"
if (BASE_MODEL_DIR / "pi05_lora_merge_manifest.json").exists():
    raise RuntimeError("ベースモデルがq/v統合版です。公平な比較のため新しいColabランタイムでこのノートブックを実行してください。")
for required in ["config.json", "model.safetensors", "policy_preprocessor.json", "policy_postprocessor.json"]:
    assert (BASE_MODEL_DIR / required).is_file(), required
print("ORIGINAL_BASE_READY", BASE_MODEL_DIR)


## q/k/v/o 実験設定

既存q/v実験と同じデータ分割、rank、学習率、stepsを使い、LoRA対象をq/k/v/oへ広げます。L4の実測余裕を使ってbatch 4とし、チェックポイントは250 stepsごとにDriveへ保存され、切断後は自動復元されます。


In [ ]:
from google.colab import drive
drive.mount("/content/drive")

TRAIN_ROOT = Path("/content/pi05_action_expert_lora_qkvo_batch4")
DRIVE_ROOT = Path("/content/drive/MyDrive/PARC2026/pi05_action_expert_lora_qkvo_batch4")
REFERENCE_ROOT = Path("/content/drive/MyDrive/PARC2026/pi05_action_expert_lora_full40")
SPLIT_MANIFEST = TRAIN_ROOT / "data_split.json"
TRAIN_OUTPUT_DIR = TRAIN_ROOT / "training"
DRIVE_CHECKPOINT_DIR = DRIVE_ROOT / "checkpoints"
VALIDATION_DIR = DRIVE_ROOT / "checkpoint_validations"
SELECTION_PATH = TRAIN_ROOT / "checkpoint_selection.json"
MERGED_MODEL_DIR = TRAIN_ROOT / "merged_model"
MERGED_VALIDATION = TRAIN_ROOT / "merged_validation.json"
QV_REFERENCE_LOSS = 0.027940072183810116

DRIVE_ROOT.mkdir(parents=True, exist_ok=True)
TRAIN_ROOT.mkdir(parents=True, exist_ok=True)
reference_manifest = REFERENCE_ROOT / "data_split.json"
drive_manifest = DRIVE_ROOT / "data_split.json"
if drive_manifest.is_file():
    shutil.copy2(drive_manifest, SPLIT_MANIFEST)
elif reference_manifest.is_file():
    shutil.copy2(reference_manifest, SPLIT_MANIFEST)
    shutil.copy2(SPLIT_MANIFEST, drive_manifest)
else:
    raise FileNotFoundError(f"q/v実験と同じ分割がありません: {reference_manifest}")
split = json.loads(SPLIT_MANIFEST.read_text(encoding="utf-8"))
assert len(split["task_counts"]) == 40
assert set(split["train_episodes"]).isdisjoint(split["validation_episodes"])
print("tasks/train/validation:", len(split["task_counts"]), len(split["train_episodes"]), len(split["validation_episodes"]))
print("Drive recovery:", DRIVE_ROOT)


In [ ]:
train_command = [
    str(PI05_PYTHON), "-u", "examples/pi05_action_expert_lora.py", "train",
    "--manifest", str(SPLIT_MANIFEST),
    "--base-model", str(BASE_MODEL_DIR),
    "--output-dir", str(TRAIN_OUTPUT_DIR),
    "--steps", "3000",
    "--save-freq", "250",
    "--preflight-steps", "5",
    "--batch-size", "4",
    "--lora-rank", "16",
    "--lora-target-profile", "qkvo",
    "--backup-dir", str(DRIVE_CHECKPOINT_DIR),
]
train_env = os.environ.copy()
train_env.update({"PYTHONUNBUFFERED": "1", "WANDB_MODE": "disabled", "WANDB_DISABLED": "true"})
print("Starting isolated q/k/v/o LoRA training", flush=True)
process = subprocess.Popen(train_command, cwd=REPO_DIR, env=train_env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
assert process.stdout is not None
for line in process.stdout:
    print(line, end="", flush=True)
returncode = process.wait()
print("training exit code:", returncode, flush=True)
if returncode != 0:
    raise RuntimeError(f"q/k/v/o training failed with exit={returncode}")


## 同一検証セットでq/vと比較

500 stepsごとの候補を同じ64サンプル・seedで比較します。現在のq/v最良loss 0.027940を下回った場合だけ統合へ進みます。


In [ ]:
select_command = [
    str(PI05_PYTHON), "-u", "examples/pi05_action_expert_lora.py", "select",
    "--manifest", str(SPLIT_MANIFEST),
    "--base-model", str(BASE_MODEL_DIR),
    "--checkpoints-dir", str(TRAIN_OUTPUT_DIR / "checkpoints"),
    "--output-dir", str(VALIDATION_DIR),
    "--output", str(SELECTION_PATH),
    "--candidate-every", "500",
    "--max-samples", "64",
]
process = subprocess.Popen(select_command, cwd=REPO_DIR, env=train_env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
assert process.stdout is not None
for line in process.stdout:
    print(line, end="", flush=True)
if process.wait() != 0:
    raise RuntimeError("q/k/v/o checkpoint selection failed")
selection = json.loads(SELECTION_PATH.read_text(encoding="utf-8"))
DRIVE_ROOT.mkdir(parents=True, exist_ok=True)  # 手動で保存先名を変えた場合にも対応
shutil.copy2(SELECTION_PATH, DRIVE_ROOT / "checkpoint_selection.json")
selected = selection["selected"]
print("base loss:", selection["base_mean_flow_matching_loss"])
print("q/v reference loss:", QV_REFERENCE_LOSS)
print("q/k/v/o best:", selected["step"], selected["mean_flow_matching_loss"])
QKVO_BEATS_QV = float(selected["mean_flow_matching_loss"]) < QV_REFERENCE_LOSS
print("QKVO_BEATS_QV:", QKVO_BEATS_QV)


In [ ]:
if not QKVO_BEATS_QV:
    print("q/k/v/oはq/vを下回らなかったため、合格済みq/vモデルを維持します。")
else:
    adapter_dir = Path(selected["adapter_dir"])
    subprocess.run([
        str(PI05_PYTHON), "-u", "examples/pi05_action_expert_lora.py", "merge",
        "--base-model", str(BASE_MODEL_DIR),
        "--adapter-dir", str(adapter_dir),
        "--output-dir", str(MERGED_MODEL_DIR),
        "--training-manifest", str(SPLIT_MANIFEST),
        "--overwrite",
    ], cwd=REPO_DIR, check=True)
    subprocess.run([
        str(PI05_PYTHON), "-u", "examples/pi05_action_expert_lora.py", "validate",
        "--manifest", str(SPLIT_MANIFEST),
        "--model-dir", str(MERGED_MODEL_DIR),
        "--output", str(MERGED_VALIDATION),
        "--max-samples", "64",
    ], cwd=REPO_DIR, check=True)
    merged_loss = float(json.loads(MERGED_VALIDATION.read_text(encoding="utf-8"))["mean_flow_matching_loss"])
    selected_loss = float(selected["mean_flow_matching_loss"])
    if abs(merged_loss - selected_loss) > max(1e-4, selected_loss * 0.05):
        raise RuntimeError("統合前後のlossが一致しません")
    DRIVE_ROOT.mkdir(parents=True, exist_ok=True)
    shutil.copy2(MERGED_VALIDATION, DRIVE_ROOT / "merged_validation.json")
    drive_merged = DRIVE_ROOT / "final/merged_model"
    if drive_merged.exists():
        shutil.rmtree(drive_merged)
    shutil.copytree(MERGED_MODEL_DIR, drive_merged)
    print("QKVO_MERGED_MODEL_READY", drive_merged)
    print("次はこの候補だけを公開4タスクで評価します。")


## 最終評価・提出ZIP・再現情報を保存

選択したq/k/v/o統合モデルを提出物へ明示的に反映し、`replan=10 / inference=10 / ensemble=False`で公開4タスクを5 episodes/task評価します。成功率に加えてsteps、軌道距離、回転量、jerk、SPARC、衝突率、時間を表示し、失敗動画とログをDriveへ保存します。評価に成功した同一モデルのZIPだけを最終版としてDriveへコピーし、Git commit・学習条件・checkpoint選択・評価結果・ZIP SHA256を再現manifestへ残します。


In [ ]:
system_packages = [
    "libosmesa6", "libgl1", "libglfw3", "libglew2.2",
    "libegl1", "libsm6", "libxext6", "libxrender1",
    "libglib2.0-0", "libmagickwand-dev", "unzip",
]
subprocess.run(["apt-get", "update", "-qq"], check=True)
subprocess.run(
    ["apt-get", "install", "-y", "-qq", "--no-install-recommends", *system_packages],
    check=True,
)
setup_env = os.environ.copy()
setup_env.update({"PYTHON": str(PI05_PYTHON), "MUJOCO_GL": "egl", "MPLBACKEND": "Agg"})
subprocess.run(["bash", "setup.sh"], cwd=REPO_DIR, env=setup_env, check=True)
EVAL_PYTHON = REPO_DIR / "venv/bin/python"
if not EVAL_PYTHON.is_file():
    raise FileNotFoundError(EVAL_PYTHON)
print("FINALIZATION_RUNTIME_READY", PI05_PYTHON, EVAL_PYTHON)


In [ ]:
if not QKVO_BEATS_QV:
    raise RuntimeError("q/k/v/o候補がq/v検証lossを下回っていないため最終化しません")
finalize_command = [
    "python", "-u", "examples/pi05_finalize_qkvo_colab.py",
    "--repo-root", str(REPO_DIR),
    "--policy-python", str(PI05_PYTHON),
    "--eval-python", str(EVAL_PYTHON),
    "--model-source", str(MERGED_MODEL_DIR),
    "--drive-root", str(DRIVE_ROOT),
    "--selection", str(SELECTION_PATH),
    "--training-manifest", str(SPLIT_MANIFEST),
    "--episodes", "5",
    "--max-steps", "600",
    "--seed", "42",
    "--replan-steps", "10",
    "--inference-steps", "10",
    "--train-steps", "3000",
    "--batch-size", "4",
    "--lora-rank", "16",
    "--record-video",
]
process = subprocess.Popen(
    finalize_command, cwd=REPO_DIR, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
    text=True, bufsize=1,
)
assert process.stdout is not None
for line in process.stdout:
    print(line, end="", flush=True)
returncode = process.wait()
if returncode != 0:
    raise RuntimeError(f"final evaluation/submission failed with exit={returncode}")


### 保存結果を確認

失敗動画を表示し、最終ZIPのパス・サイズ・SHA256と評価条件をmanifestから再確認します。Drive上の`.partial`ではなく、manifestの`drive_path`に記録された正式な`.zip`を提出します。


In [ ]:
from IPython.display import Video, display

manifest_path = DRIVE_ROOT / "final/final_reproducibility_manifest.json"
manifest = json.loads(manifest_path.read_text(encoding="utf-8"))
submission_info = manifest["submission"]
print("FINAL ZIP:", submission_info["drive_path"])
print("size GiB:", round(submission_info["size_bytes"] / 2**30, 2))
print("SHA256:", submission_info["sha256"])
print("git HEAD:", manifest["git_head"])
print("evaluation:", manifest["evaluation"]["replan_steps"], manifest["evaluation"]["inference_steps"], manifest["evaluation"]["temporal_ensemble"])
variant = "qkvo_replan_10_infer_10_ensemble_0"
video_paths = sorted((REPO_DIR / "results" / f"pi05_public_eval_{variant}" / "videos").glob("*.mp4"))
if not video_paths:
    print("失敗動画はありません（全成功、または動画保存なし）。")
for video_path in video_paths:
    print(video_path.name)
    display(Video(str(video_path), embed=True, html_attributes="controls loop"))
